# ACUARIO-IA · Fase 6: flujo de decisión profesional

Esta versión reorganiza el flujo para mejorar legibilidad: más espacio vertical, disposición por etapas, decisión en rombo, flechas que terminan en los bordes de los nodos y etiquetas claras para las ramas **SÍ/NO**. Conserva el acuario animado y el grafo de temperatura.

**Alcance académico:** temperatura simulada; rango ilustrativo de 24–26 °C; no hay sensor ni actuador físico. La corrección que se ve es una simulación, no regulación real.


In [ ]:
import math
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import FancyBboxPatch, Polygon
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

TEMP_MIN, TEMP_MAX = 24, 26

def diagnosticar(t):
    if t < TEMP_MIN: return 'ALERTA: temperatura baja', 'calentar'
    if t > TEMP_MAX: return 'ALERTA: temperatura alta', 'enfriar'
    return 'NORMAL: temperatura en rango', 'mantener'

def crear_ruta(t):
    ruta = [int(t)]
    while ruta[-1] > TEMP_MAX: ruta.append(ruta[-1] - 1)
    while ruta[-1] < TEMP_MIN: ruta.append(ruta[-1] + 1)
    return ruta

print('ACUARIO-IA listo | rango ilustrativo:', TEMP_MIN, 'a', TEMP_MAX, '°C')

## Criterios de diseño aplicados

- Flujo principal con dirección visual consistente y suficiente separación entre etapas.
- Rectángulos para procesos y rombo para la pregunta de decisión.
- Flechas con punta y etiquetas **SÍ/NO** en las ramas.
- Textos cortos, alineación uniforme y contraste alto para facilitar la lectura.
- La animación resalta una etapa a la vez, sin cambiar la lógica del sistema.


## Simulador
Elige una temperatura y pulsa **Ejecutar**. Observa el acuario, la ruta de temperatura y el **grafo de flujo**, que resalta una etapa a la vez para explicar cómo decide el sistema.

In [ ]:
control_temp = widgets.IntSlider(value=29,min=18,max=32,step=1,description='Temperatura:',continuous_update=False,style={'description_width':'initial'},layout=widgets.Layout(width='420px'))
boton = widgets.Button(description='▶ Ejecutar simulación',button_style='success',icon='play')
salida = widgets.Output()

def ejecutar(_=None):
  with salida:
    clear_output(wait=True)
    inicial=int(control_temp.value); ruta=crear_ruta(inicial); estado,accion=diagnosticar(inicial)
    print(f'Lectura inicial: {inicial} °C | Rango: {TEMP_MIN}–{TEMP_MAX} °C')
    print('Diagnóstico:',estado)
    print('Decisión:', 'mantener monitoreo' if accion=='mantener' else f'alertar; simular acción de {accion}')
    print('Ruta:', ' → '.join(f'{t}°C' for t in ruta))
    print(f'Pasos simulados: {len(ruta)-1} | Temperatura final: {ruta[-1]} °C')
    print('No hay sensores ni actuadores reales conectados.')

    fig=plt.figure(figsize=(16,11),facecolor='#071d33')
    grid=fig.add_gridspec(2,2,height_ratios=[1,1.35],width_ratios=[1,1.35],hspace=.38,wspace=.2)
    ac=fig.add_subplot(grid[0,0]); gr=fig.add_subplot(grid[0,1]); flujo=fig.add_subplot(grid[1,:])
    for ax in (ac,gr,flujo):
      ax.set_facecolor('#071d33'); ax.tick_params(colors='white')
      for sp in ax.spines.values(): sp.set_color('#35536d')

    # Acuario ilustrado con peces móviles
    ac.set_xlim(0,10); ac.set_ylim(0,10); ac.set_xticks([]); ac.set_yticks([]); ac.set_title('ACUARIO SIMULADO',color='white',fontsize=14)
    ac.fill_between([.5,8.1],[1,1],[8.5,8.5],color='#0b6388',alpha=.65)
    ac.plot([.5,.5,8.1,8.1],[1,8.5,8.5,1],color='#76d8ed',lw=2); ac.plot([.5,8.1],[1,1],color='#76d8ed',lw=2)
    for x in [1.2,2.1,7.2]: ac.plot([x,x-.2,x+.15],[1.2,2,2.7],color='#43c59e',lw=3)
    specs=[(2,5.8,'#ffb347',.42),(5,4,'#55d6d2',.35),(3.2,7,'#ff7b9c',.28)]
    bodies=[]; tails=[]; eyes=[]
    for x,y,c,size in specs:
      bodies.append(ac.scatter([x],[y],s=600*size/.4,color=c,edgecolors='white',zorder=4))
      tails.append(ac.scatter([x-.35],[y],s=150*size/.4,marker='>',color=c,zorder=3))
      eyes.append(ac.scatter([x+.12],[y+.06],s=10,color='#071d33',zorder=5))
    ac.add_patch(plt.Rectangle((8.65,2.1),.45,5.2,fill=False,edgecolor='white',lw=2))
    ac.add_patch(plt.Circle((8.875,1.95),.42,color='#f05d5e',ec='white',lw=1.5))
    merc=ac.add_patch(plt.Rectangle((8.79,2.15),.17,1,color='#f05d5e'))
    ttxt=ac.text(5,9.1,'',ha='center',color='white',fontsize=13,fontweight='bold')
    stxt=ac.text(5,.35,'',ha='center',color='#8be9fd',fontsize=10)

    # Grafo de temperaturas: nodos y ruta animada
    estados=list(range(18,33)); gr.set_xlim(17,33); gr.set_ylim(-1.1,2.2); gr.set_yticks([]); gr.set_xticks(estados[::2])
    gr.set_title('GRAFO DE ESTADOS Y TRANSICIONES (±1 °C)',color='white',fontsize=12)
    gr.set_xlabel('Cada nodo es un estado de temperatura; cada flecha representa un cambio permitido de 1 °C',color='#c6d9e6',fontsize=8)
    gr.axvspan(TEMP_MIN-.35,TEMP_MAX+.35,color='#43c59e',alpha=.18)
    # Transiciones reales del simulador: se permite avanzar o retroceder 1 °C.
    for t in estados[:-1]:
      gr.annotate('',xy=(t+.88,.08),xytext=(t+.12,.08),arrowprops={'arrowstyle':'->','color':'#49647d','lw':1.1})
      gr.annotate('',xy=(t+.12,-.08),xytext=(t+.88,-.08),arrowprops={'arrowstyle':'->','color':'#49647d','lw':1.1})
    gr.scatter(estados,[0]*len(estados),s=75,color='#49647d',edgecolors='#9fb5c7',linewidths=.6,zorder=2)
    gr.text(25,1.95,'Transición: ±1 °C por paso | Costo ilustrativo: 1 por paso',ha='center',color='#c6d9e6',fontsize=8)
    gr.text(25,-.55,'Zona objetivo ilustrativa: 24–26 °C',ha='center',color='#72f1b8',fontsize=8)
    gr.scatter([TEMP_MIN,TEMP_MAX],[0,0],s=100,color='#43c59e',edgecolors='white',zorder=3)
    linea,=gr.plot([],[],color='#36e0a1',lw=3,marker='o',markersize=7,zorder=5)
    actual=gr.scatter([],[],s=180,color='#ffd166',edgecolors='white',zorder=6)
    detalle=gr.text(25,1.55,'',ha='center',color='white',fontsize=10)

    # Flujo profesional: dirección clara, símbolos semánticos y espacio entre nodos
    flujo.set_xlim(0,14); flujo.set_ylim(0,7.4); flujo.set_xticks([]); flujo.set_yticks([])
    flujo.set_title('FLUJO DE DECISIÓN DEL SISTEMA',color='white',fontsize=14,pad=14,fontweight='bold')
    # Los procesos usan rectángulos redondeados; la decisión usa un rombo.
    etapas=[
      ('1 · LEER TEMPERATURA',(1.45,5.45)),
      ('2 · COMPARAR CON RANGO',(4.25,5.45)),
      ('3 · ¿DENTRO DEL RANGO?',(7.05,5.45)),
      ('4A · ESTADO NORMAL',(11.45,6.05)),
      ('4B · GENERAR ALERTA',(10.15,3.55)),
      ('5 · CORRECCIÓN SIMULADA',(7.05,1.65)),
      ('6 · RESULTADO FINAL',(3.55,1.65))
    ]
    cajas=[]
    for i,(label,(x,y)) in enumerate(etapas):
      if i==2:
        patch=Polygon([(x,y+.78),(x+1.25,y),(x,y-.78),(x-1.25,y)],closed=True,
                      facecolor='#17364f',edgecolor='#62839b',linewidth=1.8)
      else:
        w=2.05 if i in (0,3,4,6) else 2.25
        h=.82
        patch=FancyBboxPatch((x-w/2,y-h/2),w,h,boxstyle='round,pad=0.12,rounding_size=0.12',
                             facecolor='#17364f',edgecolor='#62839b',linewidth=1.7)
      flujo.add_patch(patch)
      flujo.text(x,y,label,ha='center',va='center',color='white',fontsize=8.5,fontweight='medium')
      cajas.append(patch)
    def flecha(a,b,curva=0):
      (x1,y1)=etapas[a][1]; (x2,y2)=etapas[b][1]
      # Shrink evita que las flechas atraviesen las figuras.
      flujo.annotate('',xy=(x2,y2),xytext=(x1,y1),
        arrowprops=dict(arrowstyle='-|>',color='#8aa9bf',lw=1.7,
                        shrinkA=34 if a!=2 else 30,shrinkB=34 if b!=2 else 30,
                        connectionstyle=f'arc3,rad={curva}'))
    flecha(0,1); flecha(1,2)
    flecha(2,3,-.08); flecha(2,4,.08)
    flecha(4,5,0); flecha(5,6,0); flecha(3,6,.22)
    flujo.text(9.05,6.25,'SÍ',color='#62e6b0',fontsize=10,fontweight='bold')
    flujo.text(9.15,4.6,'NO',color='#ff9b85',fontsize=10,fontweight='bold')
    flujo.text(7,.48,'La corrección es una simulación por pasos de 1 °C; no activa equipos físicos.',
               ha='center',color='#a9bfd0',fontsize=9)
    fase=flujo.text(7,7.0,'Preparado para iniciar',ha='center',color='#8be9fd',fontsize=9.5)
    
    # Secuencia didáctica: lectura, comparación, decisión y resultado.
    # Si la temperatura ya está en rango, omite la etapa de corrección.
    if accion=='mantener': secuencia=[0,1,2,3,6]
    else: secuencia=[0,1,2,4,5,6]
    frames=len(secuencia)*10
    def actualizar(frame):
      paso=min(frame//10,len(secuencia)-1); etapa_activa=secuencia[paso]
      for i,box in enumerate(cajas):
        box.set_facecolor('#167d75' if i==etapa_activa else '#17364f')
        box.set_edgecolor('#62e6b0' if i==etapa_activa else '#62839b')
      fase.set_text('Etapa activa: '+etapas[etapa_activa][0].replace('\n',' '))
      idx=min(max(0,frame//max(1,frames//max(1,len(ruta)))),len(ruta)-1)
      # Mantener la ruta sincronizada con la etapa de corrección
      if accion=='mantener' or etapa_activa in (0,1,2,3): idx=0
      elif etapa_activa==4: idx=0
      else: idx=min(len(ruta)-1,max(0,(frame-(secuencia.index(5)*10))//2))
      temp=ruta[idx]
      for i,(body,tail,eye,spec) in enumerate(zip(bodies,tails,eyes,specs)):
        x0,y,c,size=spec; x=.9+((x0+frame*.055*(1 if i%2==0 else -1)-.9)%6.8); yy=y+.12*math.sin(frame*.15+i)
        body.set_offsets([[x,yy]]); tail.set_offsets([[x-.35,yy]]); eye.set_offsets([[x+.12,yy+.06]])
      ttxt.set_text(f'Temperatura: {temp} °C')
      if TEMP_MIN<=temp<=TEMP_MAX: stxt.set_text('✓ EN RANGO · monitoreo'); stxt.set_color('#72f1b8')
      else: stxt.set_text('⚠ ALERTA · corrección simulada'); stxt.set_color('#ff7272')
      merc.set_height(max(.15,min(4.5,4.5*(temp-18)/14)))
      xs=ruta[:idx+1]; linea.set_data(xs,[0]*len(xs)); actual.set_offsets([[temp,0]])
      detalle.set_text(f'Ruta: {" → ".join(str(v) for v in xs)} °C')
      return bodies+tails+eyes+[merc,ttxt,stxt,linea,actual,detalle,fase]+cajas

    ani=FuncAnimation(fig,actualizar,frames=frames,interval=220,repeat=True,blit=False)
    plt.close(fig); display(HTML(ani.to_jshtml()))

boton.on_click(ejecutar)
display(widgets.VBox([control_temp,boton,salida]))
ejecutar()

## Cómo explicar esta fase

- **Lectura:** recibe un valor ficticio de temperatura.
- **Comparación:** contrasta la lectura con el intervalo configurado.
- **Decisión:** el nodo se bifurca en normal o alerta.
- **Corrección simulada:** representa pasos de 1 °C hasta entrar en rango; no acciona equipos.
- **Resultado:** muestra la temperatura final y el recorrido.

**Nota técnica:** la lógica principal es una regla de umbral. El grafo de flujo explica las decisiones; el grafo de temperaturas representa estados y transiciones. Esto no debe describirse como un modelo de aprendizaje automático.

# Fase 8 · Búsqueda de estados: BFS, UCS y A*

Esta sección añade búsqueda explícita al simulador. El modelo sigue siendo académico y simulado: no hay sensores ni actuadores físicos conectados.

## Formulación formal

- **Estado:** temperatura entera `T` en °C, entre 18 y 32.
- **Estado inicial:** valor seleccionado por el usuario.
- **Acciones:** bajar 1 °C o subir 1 °C, sin salir del intervalo modelado.
- **Prueba de meta:** `24 <= T <= 26` °C (rango ilustrativo del ejercicio).
- **Costo de cada transición:** 1 unidad; `g(n)` es la suma de los costos desde el inicio.
- **Heurística:** distancia en grados hasta el intervalo objetivo: `h(T)=24-T` si `T<24`, `h(T)=T-26` si `T>26`, y `0` si ya está en el objetivo. Es admisible para este modelo porque nunca sobreestima el número mínimo de pasos.

## Qué compara

- **BFS:** explora por niveles; con costos uniformes encuentra una ruta con el menor número de pasos.
- **UCS:** expande el estado con menor costo acumulado `g(n)`.
- **A*:** prioriza `f(n)=g(n)+h(n)`.

Como cada transición cuesta 1, UCS y BFS deben encontrar el mismo costo óptimo. A* también debe encontrar el costo óptimo con esta heurística. La cantidad de estados explorados puede variar; en un grafo tan pequeño, esta comparación es didáctica y no demuestra una ventaja universal de rendimiento.

In [ ]:
from collections import deque
import heapq
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

TEMP_MIN, TEMP_MAX = 24, 26
MODEL_MIN, MODEL_MAX = 18, 32

def es_meta(t):
    return TEMP_MIN <= t <= TEMP_MAX

def vecinos(t):
    # Transiciones bidireccionales de 1 °C dentro del espacio modelado.
    return [v for v in (t - 1, t + 1) if MODEL_MIN <= v <= MODEL_MAX]

def heuristica(t):
    # Distancia mínima en grados al intervalo objetivo.
    if t < TEMP_MIN:
        return TEMP_MIN - t
    if t > TEMP_MAX:
        return t - TEMP_MAX
    return 0

def reconstruir_padres(padres, meta):
    ruta = [meta]
    while ruta[-1] in padres:
        ruta.append(padres[ruta[-1]])
    return list(reversed(ruta))

def buscar_bfs(inicio):
    frontera = deque([inicio])
    descubiertos = {inicio}
    padres = {}
    expandidos = 0
    max_frontera = 1
    while frontera:
        actual = frontera.popleft()
        expandidos += 1
        if es_meta(actual):
            ruta = reconstruir_padres(padres, actual)
            return {'algoritmo':'BFS', 'ruta':ruta, 'costo':len(ruta)-1, 'expandidos':expandidos, 'max_frontera':max_frontera}
        for sig in vecinos(actual):
            if sig not in descubiertos:
                descubiertos.add(sig)
                padres[sig] = actual
                frontera.append(sig)
        max_frontera = max(max_frontera, len(frontera))
    return None

def buscar_ucs(inicio):
    frontera = [(0, inicio)]
    mejor_costo = {inicio: 0}
    padres = {}
    expandidos = 0
    max_frontera = 1
    while frontera:
        costo, actual = heapq.heappop(frontera)
        if costo != mejor_costo.get(actual):
            continue
        expandidos += 1
        if es_meta(actual):
            ruta = reconstruir_padres(padres, actual)
            return {'algoritmo':'UCS', 'ruta':ruta, 'costo':costo, 'expandidos':expandidos, 'max_frontera':max_frontera}
        for sig in vecinos(actual):
            nuevo_costo = costo + 1
            if nuevo_costo < mejor_costo.get(sig, float('inf')):
                mejor_costo[sig] = nuevo_costo
                padres[sig] = actual
                heapq.heappush(frontera, (nuevo_costo, sig))
        max_frontera = max(max_frontera, len(frontera))
    return None

def buscar_astar(inicio):
    frontera = [(heuristica(inicio), 0, inicio)]
    mejor_costo = {inicio: 0}
    padres = {}
    expandidos = 0
    max_frontera = 1
    while frontera:
        f, costo, actual = heapq.heappop(frontera)
        if costo != mejor_costo.get(actual):
            continue
        expandidos += 1
        if es_meta(actual):
            ruta = reconstruir_padres(padres, actual)
            return {'algoritmo':'A*', 'ruta':ruta, 'costo':costo, 'expandidos':expandidos, 'max_frontera':max_frontera}
        for sig in vecinos(actual):
            nuevo_costo = costo + 1
            if nuevo_costo < mejor_costo.get(sig, float('inf')):
                mejor_costo[sig] = nuevo_costo
                padres[sig] = actual
                heapq.heappush(frontera, (nuevo_costo + heuristica(sig), nuevo_costo, sig))
        max_frontera = max(max_frontera, len(frontera))
    return None

def comparar_busquedas(inicio):
    resultados = [buscar_bfs(inicio), buscar_ucs(inicio), buscar_astar(inicio)]
    assert all(r is not None for r in resultados), 'No se encontró una ruta a la meta.'
    costos = {r['costo'] for r in resultados}
    assert len(costos) == 1, f'Los algoritmos no coinciden en costo: {costos}'
    print(f'Estado inicial: {inicio} °C | Meta: {TEMP_MIN}–{TEMP_MAX} °C')
    print('Costo por transición: 1 | Heurística A*: distancia al rango objetivo')
    print()
    print(f"{'Algoritmo':<10} {'Ruta':<35} {'Costo':>6} {'Expandidos':>11} {'Máx. frontera':>14}")
    print('-' * 82)
    for r in resultados:
        ruta_txt = ' → '.join(f'{t}°' for t in r['ruta'])
        print(f"{r['algoritmo']:<10} {ruta_txt:<35} {r['costo']:>6} {r['expandidos']:>11} {r['max_frontera']:>14}")
    print()
    print('Interpretación: con costos unitarios, BFS y UCS encuentran costo mínimo; A* también lo garantiza con esta heurística admisible.')
    print('Los estados expandidos y el tamaño máximo de frontera son métricas del algoritmo, no una medición exacta de RAM.')

    # Visualización comparativa: cada fila muestra la ruta encontrada.
    fig, ax = plt.subplots(figsize=(13, 4.8), facecolor='#061b2b')
    ax.set_facecolor('#061b2b')
    temps = list(range(MODEL_MIN, MODEL_MAX + 1))
    ax.set_xlim(MODEL_MIN - 0.8, MODEL_MAX + 0.8)
    ax.set_ylim(-0.7, 3.0)
    ax.set_xticks(temps)
    ax.set_yticks([2, 1, 0], ['BFS', 'UCS', 'A*'])
    ax.tick_params(colors='white', labelsize=9)
    for spine in ax.spines.values(): spine.set_color('#35566c')
    ax.axvspan(TEMP_MIN - 0.45, TEMP_MAX + 0.45, color='#2bc4a2', alpha=0.2)
    ax.text((TEMP_MIN+TEMP_MAX)/2, 2.72, 'META: 24–26 °C', ha='center', color='#72f1c1', fontsize=10, fontweight='bold')
    colores = {'BFS':'#64c8ff', 'UCS':'#ffcc66', 'A*':'#66e3b1'}
    for y, r in zip([2, 1, 0], resultados):
        ax.hlines(y, MODEL_MIN, MODEL_MAX, color='#29475b', linewidth=1, zorder=1)
        ax.scatter(temps, [y]*len(temps), color='#456176', s=28, zorder=2)
        xs = r['ruta']
        ax.plot(xs, [y]*len(xs), color=colores[r['algoritmo']], linewidth=4, marker='o', markersize=7, zorder=4, label=r['algoritmo'])
        ax.scatter([inicio], [y], s=130, marker='s', color='#ff8c69', edgecolor='white', zorder=5)
        ax.scatter([xs[-1]], [y], s=160, marker='*', color='#72f1c1', edgecolor='white', zorder=6)
        ax.text(MODEL_MIN - 0.25, y + 0.22, f"{r['costo']} pasos", color=colores[r['algoritmo']], fontsize=8, ha='left')
    ax.set_title('ACUARIO-IA · Comparación de búsqueda en estados de temperatura', color='white', fontsize=13, pad=15)
    ax.set_xlabel('Temperatura (°C) · nodos conectados por cambios de ±1 °C', color='#c7d9e5')
    ax.grid(axis='x', color='#25465b', alpha=0.45)
    ax.legend(facecolor='#102d40', labelcolor='white', loc='lower right', ncol=3, framealpha=0.8)
    fig.tight_layout()
    plt.show()
    return resultados

control_inicio_busqueda = widgets.IntSlider(value=29, min=MODEL_MIN, max=MODEL_MAX, step=1, description='Temperatura inicial:', continuous_update=False, style={'description_width':'initial'}, layout=widgets.Layout(width='480px'))
boton_comparar = widgets.Button(description='Comparar BFS, UCS y A*', button_style='info', icon='search')
salida_busqueda = widgets.Output()

def ejecutar_comparacion(_=None):
    with salida_busqueda:
        clear_output(wait=True)
        comparar_busquedas(int(control_inicio_busqueda.value))

boton_comparar.on_click(ejecutar_comparacion)
display(widgets.VBox([control_inicio_busqueda, boton_comparar, salida_busqueda]))
ejecutar_comparacion()